# Easy CNN
**Authors:** Kai Herbst & Bell Wassim

### 1. Data Loading and Preprocessing
Load the dataset and perform the necessary preprocessing.

In [2]:
# Import the data loading and preprocessing function from our utility script
# Note: Ensure the .py file is in the same directory
from data_prep import load_stl10_data

# Load the dataset. The function handles downloading, splitting (80/20), 
# resizing to 96x96, normalizing, and one-hot encoding the labels.
# I am setting the batch size to 128 for efficient GPU utilization.
ds_train, ds_val, ds_test = load_stl10_data(batch_size=128)

2026-03-13 14:51:33.018021: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:984] could not open file to read NUMA node: /sys/bus/pci/devices/0000:02:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-03-13 14:51:33.018829: W tensorflow/core/common_runtime/gpu/gpu_device.cc:2251] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


Dataset split: 4000 training, 1000 validation, 8000 testing.
Data preprocessing complete!


### 2. Network Creation (Baseline Model)
Here we define the architecture of our first, simple Convolutional Neural Network (Baseline). We use a stack of Conv2D and MaxPooling2D layers for feature extraction, followed by Dense layers for classification.

In [ ]:
import keras
from keras import layers

# Input shape definition for our STL-10 images (96x96 pixels, 3 color channels)
inputs = keras.Input(shape=(96, 96, 3))

# Feature Learning Part of the CNN
x = layers.Conv2D(filters=32, kernel_size=3, activation="relu")(inputs)
x = layers.MaxPooling2D(pool_size=2)(x)
x = layers.Conv2D(filters=64, kernel_size=3, activation="relu")(x)
x = layers.MaxPooling2D(pool_size=2)(x)
x = layers.Conv2D(filters=128, kernel_size=3, activation="relu")(x)

# Classification Part
x = layers.Flatten()(x)

# Final output layer: 10 units for our 10 classes. 
outputs = layers.Dense(10, activation="softmax")(x)

# Instantiate the baseline model
model_baseline = keras.Model(inputs=inputs, outputs=outputs, name="Baseline_CNN")

# Display the architecture summary to verify the parameters
model_baseline.summary()

# Using the plot_model functionality
# keras.utils.plot_model(model_baseline, show_shapes=True, show_layer_names=True)

Model: "Baseline_CNN"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 96, 96, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 94, 94, 32)     │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 47, 47, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 45, 45, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 22, 22, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 20, 20, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 51200)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │       512,010 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 605,258 (2.31 MB)

 Trainable params: 605,258 (2.31 MB)

 Non-trainable params: 0 (0.00 B)

You must install graphviz (see instructions at https://graphviz.gitlab.io/download/) for `plot_model` to work.


### 3. Model Training
We compile the model specifying the optimizer and loss function, and then train it on our preprocessed training data. The validation data is used to monitor the model's performance on unseen data and to check for overfitting.

In [ ]:
# Compile the model: configuring the optimizer, loss function, and metrics
model_baseline.compile(
    optimizer='adam',                 # We use Adam as it is a robust standard optimizer
    loss='categorical_crossentropy',  # Categorical crossentropy is required for one-hot encoded labels
    metrics=['accuracy']              # We track accuracy to evaluate our performance
)

# Start the training process
print("Starting the training process for our baseline model...")
history_baseline = model_baseline.fit(
    ds_train,
    epochs=15,                        # 15 epochs should be enough to see if it starts overfitting
    validation_data=ds_val            # We evaluate on unseen validation data after every epoch
)